# fuselk Vision Overview

**Fusion Unified Simulation, ELM Learning & Kinetics** — the open-source autopilot for next-generation tokamaks.

This notebook maps the [VISION.md](../VISION.md) architecture to runnable code. Companion notebooks cover each pillar in depth:

| Notebook | Topic |
|----------|-------|
| `01_oil_water_barrier_math` | Coupled plasma/vapor PDEs, Peclet extraction |
| `02_helix_hqrm_diagnostics` | Phase-locked denoising, HQRM 7×7 lock |
| `03_muon_tritium_fuel_cycle` | Catalytic rate network, breeding blanket |
| `04_venturi_control_experiments` | Hierarchical RL, disruption fusion |
| `05_reactor_cell_digital_twin` | Closed-loop ReactorCell + FusionKPIs |
| **`06_unified_cross_domain_mathematics`** | **Full VISION equation sets, proofs, domain coupling** |

## The four existential bottlenecks (VISION.md)

1. **ELMs & disruptions** — MHD precursors terminate pulses
2. **Divertor melt** — localized exhaust (>10 MW/m²)
3. **Alpha sticking** — μ⁻ lost to (αμ)⁺ before catalytic breakeven
4. **Tritium self-sufficiency** — closed fuel cycle requires TBR > 1 and Pe > 1 extraction

> **Start here for theory:** `06_unified_cross_domain_mathematics.ipynb` lays out every governing equation, proof sketch, and cross-domain coupling from [VISION.md](../VISION.md). Notebooks 01–05 pair that theory with runnable experiments.


## End-to-end data flow

```
Diagnostics → HELIX/HQRM → Focal Heat Map → ELM Predictor → Venturi Controller → Actuators
                    ↓
              Physics PDE ← Breeding Blanket ← Muon Cycle
```

Each module is a Python package under `src/deepiri_fuselk/`. Experiments are registered in `experiments/registry.yaml`.


In [ ]:
import sys
from pathlib import Path

repo = Path.cwd()
if not (repo / "src" / "deepiri_fuselk").exists() and (repo.parent / "src" / "deepiri_fuselk").exists():
    repo = repo.parent

try:
    import deepiri_fuselk  # noqa: F401
except ImportError:
    sys.path.insert(0, str(repo / "src"))
    import deepiri_fuselk  # noqa: F401

import matplotlib.pyplot as plt
import numpy as np

plt.style.use("seaborn-v0_8-whitegrid")
%config InlineBackend.figure_formats = ["retina"]

import deepiri_fuselk as fuselk
print(f'fuselk v{fuselk.__version__}')

In [ ]:
registry_path = Path("experiments/registry.yaml")
if not registry_path.exists():
    registry_path = Path("../experiments/registry.yaml")

for line in registry_path.read_text().splitlines():
    if line.strip().startswith("- id:"):
        exp_id = line.split(":", 1)[1].strip()
        print(f"  • {exp_id}")


## Quick smoke test — all pillars in one cell

In [ ]:
from deepiri_fuselk.helix import HelixEngine
from deepiri_fuselk.muon import RateNetworkParams, run_rate_network
from deepiri_fuselk.physics import PDEParameters, peclet_criterion, solve_oil_water_steady
from deepiri_fuselk.sim import ReactorCell, generate_ece_shot

params = PDEParameters()
steady = solve_oil_water_steady(n_grid=32, params=params)
shot = generate_ece_shot(24, seed=0)
helix = HelixEngine().process(shot.heat_field, shot.raw_signal, shot.angles)
muon = run_rate_network(params=RateNetworkParams(R_photon=0.3), t_span=(0.0, 1e-5))
cell = ReactorCell(grid_size=16, train_elm=False)
run = cell.run(n_steps=5, seed=1)

print(f"PDE converged: {steady.converged}, Peclet Pe = {peclet_criterion(params):.2f}")
print(f"HELIX SNR: {helix.phase_locked_snr:.2f}, ELM prob: {helix.elm_probability:.2f}")
print(f"Muon fusions/μ: {muon.fusions_per_muon:.1f}, breakeven: {muon.breakeven}")
print(f"Reactor fusion score: {run.final_score:.3f}")
